Install and Import Libraries

In [33]:
!pip install shap xgboost -q

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Feature Selection
from sklearn.feature_selection import (
    SelectKBest,
    chi2,
    mutual_info_classif,
    RFE
)

# Models
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Explainable AI
import shap

# Notebook settings
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries loaded successfully.")

Libraries loaded successfully.


Upload Dataset

In [34]:
from google.colab import files
uploaded = files.upload()

Saving loan_prediction_dataset.csv to loan_prediction_dataset (2).csv


Load Dataset

In [35]:
import pandas as pd

df = pd.read_csv("loan_prediction_dataset.csv")

print("Dataset Loaded Successfully")
print("\nShape:", df.shape)

df.head()

Dataset Loaded Successfully

Shape: (2000, 7)


,Age,Income,Credit_Score,Loan_Amount,Loan_Term,Employment_Status,Loan_Approved
0,56,81788,334,15022,48,Employed,0
1,69,102879,781,21013,24,Self-Employed,1
2,46,58827,779,39687,60,Self-Employed,0
3,32,127188,364,16886,24,Unemployed,0
4,60,25655,307,26256,36,Unemployed,0


Dataset Overview

In [36]:
print("\nDataset Info:\n")
print(df.info())

print("\nMissing Values:\n")
print(df.isnull().sum())

print("\nStatistical Summary:\n")
print(df.describe())


Dataset Info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Age                2000 non-null   int64 
 1   Income             2000 non-null   int64 
 2   Credit_Score       2000 non-null   int64 
 3   Loan_Amount        2000 non-null   int64 
 4   Loan_Term          2000 non-null   int64 
 5   Employment_Status  2000 non-null   object
 6   Loan_Approved      2000 non-null   int64 
dtypes: int64(6), object(1)
memory usage: 109.5+ KB
None

Missing Values:

Age                  0
Income               0
Credit_Score         0
Loan_Amount          0
Loan_Term            0
Employment_Status    0
Loan_Approved        0
dtype: int64

Statistical Summary:

               Age         Income  Credit_Score   Loan_Amount   Loan_Term  \
count  2000.000000    2000.000000   2000.000000   2000.000000  2000.00000   
mean     43.805500   84533.58

Data Cleaning

In [37]:
# Fill numerical columns with median
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Fill categorical columns with mode
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("Missing values handled.")

Missing values handled.


Encode Categorical Features

In [38]:
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

print("Categorical variables encoded.")

df.head()

Categorical variables encoded.


,Age,Income,Credit_Score,Loan_Amount,Loan_Term,Employment_Status,Loan_Approved
0,56,81788,334,15022,48,0,0
1,69,102879,781,21013,24,1,1
2,46,58827,779,39687,60,1,0
3,32,127188,364,16886,24,2,0
4,60,25655,307,26256,36,2,0


Feature and Target Split

In [39]:
TARGET_COLUMN = "Loan_Approved"

X = df.drop(TARGET_COLUMN, axis=1)
y = df[TARGET_COLUMN]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Shape: (2000, 6)
Target Shape: (2000,)
